## Model Serving

As the class practice, the students will be required to develop local inference server using the `Churn_Modelling_train_test.csv` dataset and MLFlow for online and batch inference.

**About dataset**

This dataset is obtained from [kaggle](https://www.kaggle.com/datasets/shubhammeshram579/bank-customer-churn-prediction?resource=download). It contains information on bank customers who either left the bank or continue to be a customer. The dataset includes the following attributes:

* Customer ID: A unique identifier for each customer
* Surname: The customer's surname or last name
* Credit Score: A numerical value representing the customer's credit score
* Geography: The country where the customer resides (France, Spain or Germany)
* Gender: The customer's gender (Male or Female)
* Age: The customer's age.
* Tenure: The number of years the customer has been with the bank
* Balance: The customer's account balance
* NumOfProducts: The number of bank products the customer uses (e.g., savings account, credit card)
* HasCrCard: Whether the customer has a credit card (1 = yes, 0 = no)
* IsActiveMember: Whether the customer is an active member (1 = yes, 0 = no)
* EstimatedSalary: The estimated salary of the customer
* Exited: Whether the customer has churned (1 = yes, 0 = no)

### Model Training

For this exercise, it is necessary to have a model registered in MLFlow. Since the runs from session 2 did not include the model artifact, we will train a new model here and log it (with the artifact) so we can serve it later.

In [8]:
# import libraries
import pandas as pd
import joblib
import os
import requests
import json
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score
import mlflow
from mlflow.models import infer_signature

Start the MLflow server with the following command in the terminal: `mlflow server --host 127.0.0.1 --port 8080`.

Now we redefine the data transformation logic (same as session 2) and save it as a `.pkl` file. We will use it later for inference.

**Note:** In our pipeline from session 2 we used `pd.get_dummies` for categorical encoding (not a separate `OneHotEncoder`). So we save the whole `Transformer` class instead, which is what gets applied at inference time.

In [ ]:
# Define the Transformer class (same as session 2)
class Transformer:
    def __init__(self):
        self.DROP_COLUMNS = ['CustomerId', 'Surname', 'RowNumber']
        self.CATEGORICAL_COLS = ['Geography', 'Gender']
        self.BINARY_FEATURES = ['HasCrCard', 'IsActiveMember']

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.drop(columns=self.DROP_COLUMNS, errors='ignore')
        df = self._ensure_binary_int(df)
        df = pd.get_dummies(df, columns=self.CATEGORICAL_COLS, drop_first=False)
        return df

    def _ensure_binary_int(self, df: pd.DataFrame) -> pd.DataFrame:
        for col in self.BINARY_FEATURES:
            if col in df.columns:

                df[col] = df[col].fillna(0).astype(int)
        return df

PATH = 'model_utils/'
os.makedirs(PATH, exist_ok=True)

transformer = Transformer()
joblib.dump(transformer, f'{PATH}transformer.pkl')
print(f"Transformer saved to {PATH}transformer.pkl")

Transformer saved to model_utils/transformer.pkl


In [11]:
# Set our tracking server uri for logging
mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")
mlflow.set_experiment("Practice Experiment - Ahernandez")

2026/05/24 22:51:39 INFO mlflow.tracking.fluent: Experiment with name 'Practice Experiment - Ahernandez' does not exist. Creating a new experiment.


<Experiment: artifact_location=('file:C:/Users/ASUS/Desktop/EADA/3º trimestre/ML '
 'Operations/Activities/mlops-and-system-design/mlruns/1'), creation_time=1779655899326, experiment_id='1', last_update_time=1779655899326, lifecycle_stage='active', name='Practice Experiment - Ahernandez', tags={}, trace_location=None, workspace='default'>

Train a new Logistic Regression model and log it (with the model artifact) to MLflow. The `run_id` printed at the end is what we will use for the inference section.

In [ ]:
# Load the training dataset
df_train_full = pd.read_csv(
    r"C:\Users\ASUS\Desktop\EADA\3º trimestre\ML Operations\Activities\mlops-and-system-design\session_2\Ejercicios\Churn_Modelling_train_test.csv"
)

y = df_train_full['Exited']
X = df_train_full.drop('Exited', axis=1)


X_transformed = transformer.transform(X)
X_transformed = X_transformed.fillna(X_transformed.median())

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X_transformed, y, test_size=0.2, random_state=42, stratify=y
)

params = {"max_iter": 1000, "random_state": 42}
model = LogisticRegression(**params)
model.fit(X_train, y_train)

joblib.dump(model, f'{PATH}lr_model.pkl')
print(f"Model saved to {PATH}lr_model.pkl")

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f} | F1-score: {f1:.4f}")

Model saved to model_utils/lr_model.pkl
Accuracy: 0.8112 | F1-score: 0.3281


c:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
with mlflow.start_run() as run:
    mlflow.log_params(params)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score", f1)
    mlflow.set_tag("Training Info", "Model for serving (session 3)")

    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="churn_model",
    )

    NEW_RUN_ID = run.info.run_id


2026/05/24 22:51:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run skillful-perch-344 at: http://127.0.0.1:8080/#/experiments/1/runs/db271c07d4074702a55b55fe98dcb7ce
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


MlflowException: API request to endpoint /api/2.0/mlflow/logged-models failed with error code 404 != 200. Response body: '<!doctype html>
<html lang=en>
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you entered the URL manually please check your spelling and try again.</p>
'

### Inference

In this part, we implement functions for batch and online inference methods by providing a model uri.

We use the `RUN_ID` from the run we just created above. If you want to use a different run instead, paste its run_id manually in the variable below.

In [ ]:
# Use the run_id from the previous cell. You can override it manually if needed.
RUN_ID = NEW_RUN_ID
ARTIFACT_NAME = "churn_model"

model_uri = f"runs:/{RUN_ID}/{ARTIFACT_NAME}"
print(f"Model URI: {model_uri}")

In [ ]:

df_validation = pd.read_csv(
    r"C:\Users\ASUS\Desktop\EADA\3º trimestre\ML Operations\Activities\mlops-and-system-design\session_2\Ejercicios\Churn_Modelling_train_test.csv"
)
df_validation = df_validation.tail(500).reset_index(drop=True)
print(f"Validation dataset shape: {df_validation.shape}")
df_validation.head()

Note that the data might need to be transformed to match the model schema. You can check the schema in the `input_example.json` file in MLFlow.

In [ ]:
y_validation = df_validation['Exited']
X_validation = df_validation.drop('Exited', axis=1).copy()

X_validation[['HasCrCard', 'IsActiveMember']] = X_validation[['HasCrCard', 'IsActiveMember']].fillna(0)

X_validation_transformed = transformer.transform(X_validation)

X_validation_transformed = X_validation_transformed.fillna(X_validation_transformed.median())

expected_columns = list(X_train.columns)
X_validation_transformed = X_validation_transformed.reindex(columns=expected_columns, fill_value=0)

print(f"Shape after transformation: {X_validation_transformed.shape}")
X_validation_transformed.head()

##### Batch Inference

**Batch inference** means the model makes predictions on many examples at once (typically offline).

In [ ]:
def batch_inference(model_path: str, input: pd.DataFrame):
    """Load model from local .pkl file and predict on a batch of inputs."""
    model = joblib.load(model_path)
    predictions = model.predict(input)
    return predictions

In [ ]:
batch_prediction_result = batch_inference("model_utils/lr_model.pkl", X_validation_transformed)

print(f"First 10 predictions: {batch_prediction_result[:10]}")
print(f"Total predictions: {len(batch_prediction_result)}")

In [ ]:
cm = confusion_matrix(y_validation, batch_prediction_result)
print("Confusion Matrix:")
print(cm)
print()
print(f"Accuracy on validation: {accuracy_score(y_validation, batch_prediction_result):.4f}")
print()
print(f"True Negatives  (stayed, correct):      {cm[0,0]}")
print(f"False Positives (predicted churn, no):  {cm[0,1]}")
print(f"False Negatives (predicted no, churn):  {cm[1,0]}")
print(f"True Positives  (churn, correct):       {cm[1,1]}")

##### Online Inference

**Online inference** means the model makes predictions on demand (one request at a time), typically via an HTTP API.

Since `mlflow models serve` has a known bug with MLflow 3.x in this configuration, we use a simple **Flask server** (`server.py`) instead.

**Steps to start the server:**

1. Open a **new terminal** in VS Code and activate your venv:  
   `.\venv\Scripts\Activate.ps1`
2. Make sure Flask is installed: `pip install flask`
3. Navigate to the exercise folder:  
   `cd "session_3/Class Exercise"`
4. Start the Flask server:  
   `python server.py`

Once the terminal prints `Running on http://127.0.0.1:5000`, you can run the cells below.

In [ ]:
import requests
import json

In [ ]:
# import validation dataset to test inference - just one record
df_validation_one = pd.read_csv(
    r"C:\Users\ASUS\Desktop\EADA\3º trimestre\ML Operations\Activities\mlops-and-system-design\session_2\Ejercicios\Churn_Modelling_train_test.csv"
).head(1)
print("Real label:", df_validation_one['Exited'].values[0])
df_validation_one = df_validation_one.drop('Exited', axis=1)
df_validation_one

Transform the data so that it matches the model schema (same transformer used during training).

In [ ]:
# transform data - same approach as the batch validation cell
df_validation_one_copy = df_validation_one.copy()
df_validation_one_copy[['HasCrCard', 'IsActiveMember']] = df_validation_one_copy[['HasCrCard', 'IsActiveMember']].fillna(0)

df_validation_one_transformed = transformer.transform(df_validation_one_copy)
df_validation_one_transformed = df_validation_one_transformed.fillna(df_validation_one_transformed.median())
df_validation_one_transformed = df_validation_one_transformed.reindex(columns=expected_columns, fill_value=0)

df_validation_one_transformed

In [ ]:
def get_inference_endpoint(host="http://127.0.0.1", port=5000):
    return f"{host}:{port}/invocations"

url = get_inference_endpoint()
print(f"Inference URL: {url}")

In [ ]:
# define a function to implement online inference - pandas input
def online_inference_pandas(url: str, input: pd.DataFrame):
    """Send a pandas DataFrame to the Flask inference endpoint."""
    headers = {"Content-Type": "application/json"}
    # Use pandas to_json so numpy types (float64, int64, bool) are serialized correctly.
    # json.dumps with default=str would convert numbers to strings, breaking the model.
    split_dict = json.loads(input.to_json(orient="split"))
    payload = {"dataframe_split": split_dict}
    response = requests.post(url, headers=headers, data=json.dumps(payload))
    return response

In [ ]:
response_pandas = online_inference_pandas(url=url, input=df_validation_one_transformed)
print("Status code:", response_pandas.status_code)
print("Response content:", response_pandas.content)

In [ ]:
# define a function to implement online inference with mlflow - json input
def online_inference_json(url: str, input: dict):
    """Send a raw JSON payload (dataframe_split format) to the MLflow inference endpoint."""
    headers = {"Content-Type": "application/json"}
    response = requests.post(url, headers=headers, data=json.dumps(input, default=str))
    return response

In [ ]:
# define the json as required by MLFlow, built from our single transformed record
input_json = {
    "dataframe_split": {
        "columns": list(df_validation_one_transformed.columns),
        "data": df_validation_one_transformed.values.tolist()
    }
}

response_json = online_inference_json(url=url, input=input_json)
print("Status code:", response_json.status_code)
print("Response content:", response_json.content)